# RockWatch - Phase 2: Exploratory Data Analysis (EDA)
**Project:** Intelligent Landslide Susceptibility Prediction System for Himachal Pradesh  
**Objective:** Explore feature distributions, spatial splits, predictive separation power, and multicollinearity.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_auc_score

# Set project root
PROJECT_ROOT = Path('.').resolve().parents[0] if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_processed_data
from src.visualisation.eda_plots import (
    plot_class_distribution,
    plot_feature_distributions,
    plot_correlation_matrix,
    plot_lulc_distribution,
    plot_elevation_slope_profile,
)
from src.visualisation.spatial_plots import plot_spatial_splits

## 1. Load Processed Datasets

In [ ]:
train_spatial = load_processed_data('train', features_only=False, processed_dir=PROJECT_ROOT / 'data' / 'processed')
val_spatial = load_processed_data('val', features_only=False, processed_dir=PROJECT_ROOT / 'data' / 'processed')
test_spatial = load_processed_data('test', features_only=False, processed_dir=PROJECT_ROOT / 'data' / 'processed')

f_train = load_processed_data('train', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_val = load_processed_data('val', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')
f_test = load_processed_data('test', features_only=True, processed_dir=PROJECT_ROOT / 'data' / 'processed')

print(f'Train rows: {len(train_spatial):,} | Val rows: {len(val_spatial):,} | Test rows: {len(test_spatial):,}')
print(f'Features schema: {list(f_train.columns)}')

## 2. Class Distribution Across Spatial Splits

In [ ]:
fig1 = plot_class_distribution(train_spatial, val_spatial, test_spatial)
plt.show()

## 3. Geographic Partitioning Map

In [ ]:
fig2 = plot_spatial_splits(train_spatial, val_spatial, test_spatial)
plt.show()

## 4. Continuous Feature Distributions by Class

In [ ]:
continuous_features = [
    'Slope_z_corr', 'Elevation_z_corr', 'NDVI_z_corr', 'Rainfall_z_corr',
    'Dist_River_z_corr', 'FlowAccumulation_z_corr', 'sin_aspect', 'cos_aspect'
]
fig3 = plot_feature_distributions(f_train, continuous_features, target_col='class')
plt.show()

## 5. Feature Predictive Separation Power (Univariate Ranking)

In [ ]:
stats_list = []
c1 = f_train[f_train['class'] == 1]
c0 = f_train[f_train['class'] == 0]

for feat in [c for c in f_train.columns if c != 'class']:
    vals1 = c1[feat].dropna()
    vals0 = c0[feat].dropna()
    u_stat, p_val = stats.mannwhitneyu(vals1, vals0, alternative='two-sided')
    mean_diff = vals1.mean() - vals0.mean()
    pooled_std = np.sqrt(((len(vals1) - 1) * vals1.var() + (len(vals0) - 1) * vals0.var()) / (len(vals1) + len(vals0) - 2))
    cohen_d = mean_diff / pooled_std if pooled_std > 0 else 0.0
    auc = roc_auc_score(f_train['class'], f_train[feat])
    if auc < 0.5:
        auc = 1.0 - auc
    stats_list.append({
        'Feature': feat,
        'Mean_Class1': vals1.mean(),
        'Mean_Class0': vals0.mean(),
        'Delta': mean_diff,
        'Cohen_d': cohen_d,
        'P_Value': p_val,
        'Univariate_AUC': auc,
    })

stats_df = pd.DataFrame(stats_list).sort_values('Univariate_AUC', ascending=False)
stats_df

## 6. Correlation Heatmap & Multicollinearity Assessment

In [ ]:
fig4 = plot_correlation_matrix(f_train, continuous_features + ['class'])
plt.show()

## 7. Land Use / Land Cover (LULC) Susceptibility Analysis

In [ ]:
fig5 = plot_lulc_distribution(train_spatial)
plt.show()

## 8. Elevation vs Slope Profile

In [ ]:
fig6 = plot_elevation_slope_profile(train_spatial)
plt.show()